In [1]:
%run ../../../imports_common.py -e "encoding-stf"

In [2]:
import  numpy  as np
import  pandas as pd
from    typing                 import Any, Dict, List, Union
from    IPython.display        import display
from    sentence_transformers  import SentenceTransformer
# package modules
from    velari_core.core       import read_cache_dir

# Encoding text with `sentence-transformers`

Encode text directly with `sentence_transformers` (no gateway) and attach the vectors to a DataFrame.

**Choosing a model** — pick by language, task, and size/speed. Vectors are `float` lists of the listed dimension:

| Model | Dim | Max tokens | Params | Latency (ms/text) | Best for |
|---|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 256 | ~23M | 0.18 | smaller and faster; good general-purpose default for prototyping |
| `all-MiniLM-L12-v2` | 384 | 128 | ~33M | 0.35 | deeper MiniLM variant; general-purpose, but a shorter input limit |
| `all-mpnet-base-v2` | 768 | 384 | ~109M | 0.56 | better general-purpose quality; slower, larger vectors |
| `multi-qa-MiniLM-L6-cos-v1` | 384 | 512 | ~23M | 0.21 | semantic search: short questions matched to longer passages |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 128 | ~118M | 0.31 | multilingual text (50+ languages); paraphrase/similarity |
| `BAAI/bge-small-en-v1.5` | 384 | 512 | ~33M | 0.34 | English retrieval; small and fast |
| `BAAI/bge-base-en-v1.5` | 768 | 512 | ~109M | 0.63 | English retrieval; stronger, slower |
| `BAAI/bge-m3` | 1024 | 8192 | ~568M | 1.77 | long documents and multilingual retrieval; much larger and slower |

*Latency* is the median per-text encode time on an Apple-silicon GPU (MPS): 64 short ticket sentences, batch size 32, median of 5 runs after a warm-up. Treat it as relative — it varies with hardware, batch size, and text length. Inputs longer than *Max tokens* are truncated. Vectors from different models aren't comparable — index and query with the same one.

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=read_cache_dir(app="huggingface/hub"))
enc: Dict[str, Any] = dict(normalize_embeddings=True, batch_size=32, show_progress_bar=False)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
def to_frame(
    data:     Union[List[str], np.ndarray, pd.DataFrame],
    vectors:  np.ndarray,
    text_col: str = "text",
    emb_col:  str = "emb",
) -> pd.DataFrame:
    """Attach encoded vectors to their texts (list, array, or DataFrame) as a column of lists.

    Args:
        data (Union[List[str], np.ndarray, pd.DataFrame]): Encoded texts; a DataFrame keeps its columns and isn't mutated.
        vectors (np.ndarray): `model.encode()` output, one row per text.
        text_col (str): Text column name when `data` isn't a DataFrame.
        emb_col (str): Column that receives one `List[float]` per record.

    Returns:
        pd.DataFrame: `data` (or a new text column) plus `emb_col`.

    Examples:
        >>> df_emb = to_frame(tickets, model.encode(tickets, normalize_embeddings=True))
    """
    df_base = data if isinstance(data, pd.DataFrame) else pd.DataFrame({text_col: [str(t) for t in data]})
    return df_base.assign(**{emb_col: vectors.tolist()})

## List or array input

In [5]:
tickets = [
    "Customer cannot reset billing portal password",
    "Invoice ACC-10293 was charged twice this month",
    "How do I update my payment method?",
    "Export the Q3 churn analysis report to CSV",
]
vectors = model.encode(tickets, **enc)                    # ndarray (n, dim); np.array(tickets) works too
list_emb = vectors.tolist()                               # plain list of vectors
df_emb = to_frame(tickets, vectors)                       # DataFrame: text, emb
df_arr_emb = to_frame(np.array(tickets), vectors, text_col="ticket")
display(df_emb)
print(vectors.shape, type(list_emb), type(df_emb["emb"][0]))

,text,emb
0,Customer cannot reset billing portal password,"[-0.017588522285223007, 0.02661869116127491, -..."
1,Invoice ACC-10293 was charged twice this month,"[-0.07946699112653732, -0.00025136754265986383..."
2,How do I update my payment method?,"[0.032165948301553726, -0.0028027896769344807,..."
3,Export the Q3 churn analysis report to CSV,"[-0.024230267852544785, -0.017546826973557472,..."


(4, 384) <class 'list'> <class 'list'>


## DataFrame input

In [6]:
df_tickets = pd.DataFrame({"ticket_id": ["T-101", "T-102", "T-103", "T-104"], "text": tickets})
df_emb = to_frame(df_tickets, model.encode(df_tickets["text"].tolist(), **enc))                          # default "emb"
df_emb = to_frame(df_emb, model.encode(df_tickets["text"].tolist(), **enc), emb_col="text_emb")          # custom name
print(df_emb.columns.tolist(), "emb" in df_tickets.columns)

['ticket_id', 'text', 'emb', 'text_emb'] False


## Preprocessing

`model.preprocess` runs the tokenization step of `encode` (truncating to `model.max_seq_length`) and returns tensors: `input_ids`, `attention_mask`, and so on. It takes a list of texts, so convert an array with `.tolist()`. Summing `attention_mask` gives the token count per text, which shows what gets truncated before you encode.

In [7]:
long_ticket = " ".join(["invoice"] * 300)
texts       = tickets + [long_ticket]
features    = model.preprocess(texts)
df_tokens   = (
    pd.DataFrame({"text": [t[:40] for t in texts], "n_tokens": features["attention_mask"].sum(dim=1).tolist()})
    .assign(truncated=lambda d: d["n_tokens"] >= model.max_seq_length)
)
display(df_tokens)

,text,n_tokens,truncated
0,Customer cannot reset billing portal pas,8,False
1,Invoice ACC-10293 was charged twice this,15,False
2,How do I update my payment method?,10,False
3,Export the Q3 churn analysis report to C,13,False
4,invoice invoice invoice invoice invoice,256,True


Embeddings are normalized, so a dot product is cosine similarity:

In [8]:
ids = df_emb["ticket_id"].tolist()
pd.DataFrame(vectors @ vectors.T, index=ids, columns=ids).round(2)

,T-101,T-102,T-103,T-104
T-101,1.00,0.38,0.34,0.06
T-102,0.38,1.00,0.28,0.04
T-103,0.34,0.28,1.00,-0.02
T-104,0.06,0.04,-0.02,1.00
